In [1]:
# Tecnología
import json
import calendar
import pandas as pd
import numpy as np
from sparky_bc import Sparky
import datetime as dt
from dateutil.relativedelta import relativedelta
from dateutil.parser import isoparse

# files lz conection
path_sparky_conf = '/Users/santlond/Documents/sparky_conf.json'

# Configurar conexión a LZ
with open(path_sparky_conf, 'rb') as JSON_lz_File:
    sp_config = json.loads(JSON_lz_File.read())
    
USER='santlond'
PASS=sp_config['ID']
DSN='IMPALA_PROD'
LOGDIR= 'logs'
# sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp")
sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp", spark_submit="spark3-submit")
 
# sparky = Sparky(username=USER, password=PASS, dsn=DSN)

helper = sparky.helper

/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-02-09 08:23:13 - [WARNING] - No se encontro la carpeta "/Users/santlond/Documents/ADQUIRENCIA_FERIA_EVA/atribucion/enfoque_huella_digital/logs" para guardar los logs


 ____  _____ __  __  ___ _____ _____ 
|  _ \| ____|  \/  |/ _ \_   _| ____|
| |_) |  _| | |\/| | | | || | |  _|  
|  _ <| |___| |  | | |_| || | | |___ 
|_| \_\_____|_|  |_|\___/ |_| |_____|
                                     
 ____  ____   _    ____  _  __
/ ___||  _ \ / \  |  _ \| |/ /
\___ \| |_) / _ \ | |_) | ' / 
 ___) |  __/ ___ \|  _ <| . \ 
|____/|_| /_/   \_\_| \_\_|\_\
                              



# Adquirencia

## Datos huella digital

In [60]:
# Leer datos
df = pd.read_csv('datos/formularios_adquirencia_enviar_datos.csv', converters={'fecha':pd.Timestamp})

In [58]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2357 entries, 0 to 2356
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   user_id   263 non-null    object        
 1   campaign  2335 non-null   object        
 2   medium    2352 non-null   object        
 3   source    2357 non-null   object        
 4   fecha     2357 non-null   datetime64[ns]
dtypes: datetime64[ns](1), object(4)
memory usage: 92.2+ KB


In [56]:
# Registros sin user_id
df.user_id.isna().sum()

2094

In [29]:
# Número de Registros con user_id
2357-2094

263

In [30]:
# Registros con user_id
df[~df.user_id.isna()]

,user_id,campaign,medium,source,fecha
5,110402160708000,(direct),(none),(direct),2025-09-18
10,50727082517000,(direct),(none),(direct),2025-09-19
11,190509132722000,(direct),(none),(direct),2025-09-19
12,50727082517000,(direct),(none),(direct),2025-09-19
13,50727082517000,(direct),(none),(direct),2025-09-19
...,...,...,...,...,...
2344,61022102853653,(direct),(none),(direct),2026-01-29
2346,170711153433002,(organic),organic,google,2026-01-29
2352,210226164951000,(organic),organic,google,2026-01-29
2353,120816131157000,(direct),(none),(direct),2026-01-29


In [31]:
# Rango de fechas de los registros
df.fecha.min(), df.fecha.max()

(Timestamp('2025-09-18 00:00:00'), Timestamp('2026-01-29 00:00:00'))

In [61]:
# Dataframe registros con user_id
df_user_id = df[~df.user_id.isna()].reset_index().drop(columns='index')
df_user_id['contador'] = 1
df_user_id['source_medium'] = df_user_id['source'] + ' / ' + df_user_id['medium']
df_user_id

,user_id,campaign,medium,source,fecha,contador,source_medium
0,110402160708000,(direct),(none),(direct),2025-09-18,1,(direct) / (none)
1,50727082517000,(direct),(none),(direct),2025-09-19,1,(direct) / (none)
2,190509132722000,(direct),(none),(direct),2025-09-19,1,(direct) / (none)
3,50727082517000,(direct),(none),(direct),2025-09-19,1,(direct) / (none)
4,50727082517000,(direct),(none),(direct),2025-09-19,1,(direct) / (none)
...,...,...,...,...,...,...,...
258,61022102853653,(direct),(none),(direct),2026-01-29,1,(direct) / (none)
259,170711153433002,(organic),organic,google,2026-01-29,1,google / organic
260,210226164951000,(organic),organic,google,2026-01-29,1,google / organic
261,120816131157000,(direct),(none),(direct),2026-01-29,1,(direct) / (none)


In [101]:
df_last_click = df_user_id.groupby('user_id')['source_medium', 'fecha'].max().reset_index()
df_last_click.source_medium.fillna('canal_nulo', inplace=True)
df_last_click

/var/folders/gt/z1ffyd2d3qdgxgy_874364xc0000gr/T/ipykernel_10205/1782126945.py:1: FutureWarning: Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.
  df_last_click = df_user_id.groupby('user_id')['source_medium', 'fecha'].max().reset_index()


,user_id,source_medium,fecha
0,100210111255000,google / organic,2025-11-22
1,100321224557006,(direct) / (none),2025-11-22
2,100805183512000,(direct) / (none),2025-11-22
3,100903121103000,(direct) / (none),2025-12-09
4,100923095037000,(direct) / (none),2025-11-22
...,...,...,...
153,980629181532000,google / organic,2025-11-22
154,980629190643017,facebook / cpc,2025-12-12
155,991129145129000,(direct) / (none),2025-09-26
156,ccd3356ea3eb297a82c6176e31fde6e90cf58606a4ad42...,QR / Empaque,2025-10-15


In [102]:
df_last_click.groupby('source_medium')['user_id'].count().reset_index().sort_values(by='user_id', ascending=False).reset_index(drop=True)

,source_medium,user_id
0,(direct) / (none),90
1,google / organic,27
2,google / cpc,9
3,Organic / Redirect,5
4,email / link,4
5,facebook / cpc,4
6,canal_nulo,3
7,pse.todo1.com / referral,3
8,app-bancolombia / link,2
9,app / icono,2


In [103]:
# Crear nueva categoria, refleja el canal de adquisición
dict_categorias = {
    '(direct) / (none)': ['directo'],
    'google / organic': ['organico'],
    'Organic / Redirect': ['organico'],
    'google / cpc': ['google'],
    'facebook / cpc': ['meta'],
    'email / wtrack': ['email'],
    'email / link': ['email'],
    'app / card': ['app'],
    'MiB / App': ['app'],
    'app / icono': ['app'],
    'app / link': ['app'],
    'app / banner': ['app'],
    'pse.todo1.com / referral': ['referidos_otros'],
    'forms.office.com / referral': ['referidos_otros'],
    'leasing.grupobancolombia.com / referral': ['medios_propios'],
    'lm.facebook.com / referral': ['medios_propios'],
    'QR / Empaque': ['sin_clasificar']
}

df_last_click['canal'] = df_last_click.source_medium.replace(dict_categorias)
df_last_click



,user_id,source_medium,fecha,canal
0,100210111255000,google / organic,2025-11-22,organico
1,100321224557006,(direct) / (none),2025-11-22,directo
2,100805183512000,(direct) / (none),2025-11-22,directo
3,100903121103000,(direct) / (none),2025-12-09,directo
4,100923095037000,(direct) / (none),2025-11-22,directo
...,...,...,...,...
153,980629181532000,google / organic,2025-11-22,organico
154,980629190643017,facebook / cpc,2025-12-12,meta
155,991129145129000,(direct) / (none),2025-09-26,directo
156,ccd3356ea3eb297a82c6176e31fde6e90cf58606a4ad42...,QR / Empaque,2025-10-15,sin_clasificar


In [ ]:
# Distribuciónd de frecuencia de longitud de user_id
df_last_click.user_id.str.len().value_counts()

15     114
14      40
13       1
12       1
128      1
36       1
Name: user_id, dtype: int64

In [122]:
# Se verifica que los user_id con longitud mayor a 15 son inválidos, contienen caracteres no numéricos
df_last_click = df_last_click[df_last_click.user_id.str.len() <= 15]
df_last_click

,user_id,source_medium,fecha,canal
0,100210111255000,google / organic,2025-11-22,organico
1,100321224557006,(direct) / (none),2025-11-22,directo
2,100805183512000,(direct) / (none),2025-11-22,directo
3,100903121103000,(direct) / (none),2025-12-09,directo
4,100923095037000,(direct) / (none),2025-11-22,directo
...,...,...,...,...
151,980629003706002,google / organic,2025-10-17,organico
152,980629071746019,google / organic,2026-01-23,organico
153,980629181532000,google / organic,2025-11-22,organico
154,980629190643017,facebook / cpc,2025-12-12,meta


## Datos Vinculaciones

In [3]:
dict_ult_ing_adqu_vinc = helper.obtener_ultima_ingestion('resultados_vspc_medios_de_pago.gsap_m_comercios')
dict_ult_ing_adqu_vinc

2026-02-04 15:12:40 - [INFO] - Buscando fechas para resultados_vspc_medios_de_pago.gsap_m_comercios
2026-02-04 15:12:40 - [INFO] - Transcurrido: 1770235960, Tiempo de Refresco = 1000
/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:476: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, cn)
2026-02-04 15:12:43 - [INFO] - Finalizo la busqueda, duracion: 00:02.8, resultado: {'year': 2026, 'month': 2, 'day': 4}


{'year': 2026, 'month': 2, 'day': 4}

# Wompi

## Datos huella digital

In [166]:
df_wompi = pd.read_excel('datos/Wompi - atribución base_5x_excel 1.xlsx', sheet_name='base_5x')
df_wompi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120158 entries, 0 to 120157
Data columns (total 16 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   user_id                      120158 non-null  object 
 1   email                        120158 non-null  object 
 2   phone_number                 120158 non-null  int64  
 3   merchant_name                53619 non-null   object 
 4   user_name                    120158 non-null  object 
 5   legal_id_type                37484 non-null   object 
 6   legal_id                     53619 non-null   float64
 7   locale                       120158 non-null  object 
 8   contactKey                   0 non-null       float64
 9   is_gateway                   99860 non-null   object 
 10  is_company                   0 non-null       float64
 11  onboarding_status            120158 non-null  object 
 12  created_at                   118708 non-null  object 
 13 

In [167]:

df_wompi.head()

,user_id,email,phone_number,merchant_name,user_name,legal_id_type,legal_id,locale,contactKey,is_gateway,is_company,onboarding_status,created_at,updated_at,signup_campaign,chosen_plan (cuando aplica)
0,36e33065-8eef-4b7b-94d8-9020d0c9d5ce,orlandoalfonsofrancovergara@gmail.com,3223923330,NaN,judana31,NaN,NaN,co,NaN,False,NaN,delete-merchant-noti-send,NaN,2025-07-29T05:01:02.094Z,NaN,NaN
1,84d3027e-57f7-4825-8998-ba89cf0db6c5,misifubabykids@gmail.com,3102482597,Misifu baby kids,mnaranjo85,CC,5.390758e+07,co,NaN,False,NaN,delete-fe-erratas-mail,NaN,NaN,NaN,NaN
2,7ed722d2-292b-4a23-bb99-5e90d500e7e3,dalejandromartinezg@gmial.com,3137214186,NaN,alejandro31,NaN,NaN,co,NaN,NaN,NaN,delete-fe-erratas-mail,NaN,NaN,NaN,NaN
3,0d6b0ad5-8faa-4e4b-9494-aec66a1b65c9,julianr-1215@hotmail.com,3023642556,Barberia Raices,riascos20,CC,1.130652e+09,co,NaN,False,NaN,delete-fe-erratas-mail,NaN,NaN,NaN,NaN
4,28db35fa-3a84-42b6-aa75-176185eba1a7,plataforma@grppk.com,3156341478,NaN,gruppok00,NaN,NaN,co,NaN,NaN,NaN,delete-fe-erratas-mail,NaN,NaN,NaN,NaN


In [168]:
# Tranformar fecha a formato datetime
df_wompi['fecha'] = pd.to_datetime(df_wompi.updated_at, utc=True)
df_wompi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120158 entries, 0 to 120157
Data columns (total 17 columns):
 #   Column                       Non-Null Count   Dtype              
---  ------                       --------------   -----              
 0   user_id                      120158 non-null  object             
 1   email                        120158 non-null  object             
 2   phone_number                 120158 non-null  int64              
 3   merchant_name                53619 non-null   object             
 4   user_name                    120158 non-null  object             
 5   legal_id_type                37484 non-null   object             
 6   legal_id                     53619 non-null   float64            
 7   locale                       120158 non-null  object             
 8   contactKey                   0 non-null       float64            
 9   is_gateway                   99860 non-null   object             
 10  is_company                   0 n

In [169]:
df_wompi.fecha.min(), df_wompi.fecha.max()

(Timestamp('2025-04-28 20:43:55.875000+0000', tz='UTC'),
 Timestamp('2026-02-02 19:53:03.216000+0000', tz='UTC'))

In [170]:
# Seleccionar quienes diligenciaron completamente el formularios
df_wompi = df_wompi[df_wompi['onboarding_status'].isin(['pending-approval-wompi', 'accept-terms-and-conditions', 'business-approved'])]
df_wompi.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 35841 entries, 45 to 120153
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype              
---  ------                       --------------  -----              
 0   user_id                      35841 non-null  object             
 1   email                        35841 non-null  object             
 2   phone_number                 35841 non-null  int64              
 3   merchant_name                34978 non-null  object             
 4   user_name                    35841 non-null  object             
 5   legal_id_type                24460 non-null  object             
 6   legal_id                     34978 non-null  float64            
 7   locale                       35841 non-null  object             
 8   contactKey                   0 non-null      float64            
 9   is_gateway                   35231 non-null  object             
 10  is_company                   0 non-null     

In [171]:
# Crear nueva categoria, refleja el canal de adquisición
dict_categorias = {
    'default': ['directo'],
    'bancolombia-sem': ['google'],
    'BANCOLOMBIA_PSN_AON_SEM_PEF_CPA_CONVERSION_WOMPI_WOMPI_CT_MDO00885_C106505_001': ['google'],
    'BANCOLOMBIA_PSN_AON_SEM_PEF_CPA_CONVERSION_WOMPI_WOMPI_CQ_MDO00885_C106505_001': ['google'],
    'BANCOLOMBIA_OTROS_AON_GG_PEF_CPA_CONVERSION__WOMPI-COMERCIOS_MDO00885_26_026': ['google'],
    'WOMPI_AON_FB_CPA_CONVERSION_REGISTROS-NUEVOS_ALWAYS-ON_VINCULACION-3': ['meta'],
    'ESTRATEGIA-5X-PRIMER-MAIL': ['email'],
    'ESTRATEGIA-5X-SEGUNDO-MAIL': ['email'],
    'REGISTRO_APP_WOMPI': ['app'],
    'correspondentBank': ['sin_clasificar'],
    'massEnrollment': ['sin_clasificar']
}

df_wompi['canal'] = df_wompi.signup_campaign.replace(dict_categorias)
df_wompi

,user_id,email,phone_number,merchant_name,user_name,legal_id_type,legal_id,locale,contactKey,is_gateway,is_company,onboarding_status,created_at,updated_at,signup_campaign,chosen_plan (cuando aplica),fecha,canal
45,6bef02b4-170a-4b0f-9faa-6cc2e7e1053c,spardo@cerips.com,3123645954,CLINICA CER IPS,cerips01,NaN,9.019342e+08,co,NaN,False,NaN,pending-approval-wompi,NaN,2025-09-11T22:22:49.422Z,NaN,NaN,2025-09-11 22:22:49.422000+00:00,NaN
51,29be6587-9e17-4275-948f-06b3604a118e,info@mangomascotas.com,3167799609,Mango Mascotas,mangomascotas17,CC,1.114402e+09,co,NaN,False,NaN,pending-approval-wompi,NaN,NaN,NaN,NaN,NaT,NaN
83,726ba732-ca49-4c03-a6f3-e995ad87c735,comercializadoranwll@gmail.com,3103188817,COMERCIALIZADORA NACIONAL WLL LTDA,comercializadora14,NaN,8.301251e+08,co,NaN,False,NaN,pending-approval-wompi,NaN,2025-07-22T02:28:19.301Z,NaN,NaN,2025-07-22 02:28:19.301000+00:00,NaN
88,a3ae2a36-53d5-48a7-8e9c-ae1e8ec3a097,luisaalvarez2280@gmail.com,3124958125,Maquillaje Cosmico,luisacosmico221,CC,1.152706e+09,co,NaN,False,NaN,pending-approval-wompi,NaN,2025-07-24T01:24:32.955Z,NaN,NaN,2025-07-24 01:24:32.955000+00:00,NaN
154,29ba7bc2-2148-4e48-bb82-e8c8e3fd3077,adr_system@hotmail.com,3173832192,CRISTALERIA UNIVERSAL RIMAX,adrianagarcia03,CC,5.231940e+07,co,NaN,False,NaN,pending-approval-wompi,2025-07-31T22:11:41.453Z,2025-07-31T22:16:28.839Z,default,NaN,2025-07-31 22:16:28.839000+00:00,directo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120141,43808114-496d-4682-94d6-01a4cb4b1cc6,lorenacortes1104@gmail.com,3147973393,SaLu Accesorios,lorenacortes11,CC,1.054572e+09,co,NaN,False,NaN,business-approved,2026-01-27T19:00:01.095Z,2026-01-27T19:12:52.266Z,default,nano_micro_special_plan,2026-01-27 19:12:52.266000+00:00,directo
120143,13d125cc-8566-4590-b814-a861dc7d803c,angiecoronado909@gmail.com,3042175460,Optikas tokio,optikastokio04,CC,1.024502e+09,co,NaN,False,NaN,business-approved,2026-01-27T19:02:04.463Z,2026-01-27T19:14:28.947Z,default,nano_micro_special_plan,2026-01-27 19:14:28.947000+00:00,directo
120145,f1729cc2-c1db-45cb-b0ab-f4ff3ff59e14,jonathanagudeloarango@gmail.com,3128842920,NaN,cbjonaxef025,NaN,NaN,co,NaN,NaN,NaN,business-approved,2026-01-27T19:02:32.296Z,2026-01-27T19:04:55.559Z,correspondentBank,advanced_cb_freemium_plan,2026-01-27 19:04:55.559000+00:00,sin_clasificar
120152,4ddfdfba-1b43-4b5f-967c-6a157f8a7e4d,jaimemarin7023@gmail.com,3217463079,Pctecno,pctecno70,CC,7.069733e+07,co,NaN,False,NaN,business-approved,2026-01-27T19:10:08.574Z,2026-01-27T19:16:11.853Z,REGISTRO_APP_WOMPI,nano_micro_special_plan,2026-01-27 19:16:11.853000+00:00,app


In [172]:
# Eliminar registros sin canal asignado
df_wompi = df_wompi[~df_wompi.canal.isna()]

In [173]:
df_wompi.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 35728 entries, 154 to 120153
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype              
---  ------                       --------------  -----              
 0   user_id                      35728 non-null  object             
 1   email                        35728 non-null  object             
 2   phone_number                 35728 non-null  int64              
 3   merchant_name                34867 non-null  object             
 4   user_name                    35728 non-null  object             
 5   legal_id_type                24421 non-null  object             
 6   legal_id                     34867 non-null  float64            
 7   locale                       35728 non-null  object             
 8   contactKey                   0 non-null      float64            
 9   is_gateway                   35118 non-null  object             
 10  is_company                   0 non-null    

In [174]:
# Seleccionar registros con canal impulsados por marketing [google, meta, email]
df_wompi = df_wompi[df_wompi.canal.isin(['google', 'meta', 'email'])]
df_wompi.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 3674 entries, 838 to 119739
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype              
---  ------                       --------------  -----              
 0   user_id                      3674 non-null   object             
 1   email                        3674 non-null   object             
 2   phone_number                 3674 non-null   int64              
 3   merchant_name                3674 non-null   object             
 4   user_name                    3674 non-null   object             
 5   legal_id_type                2750 non-null   object             
 6   legal_id                     3674 non-null   float64            
 7   locale                       3674 non-null   object             
 8   contactKey                   0 non-null      float64            
 9   is_gateway                   3674 non-null   object             
 10  is_company                   0 non-null     

In [175]:
df_last_click = df_wompi.groupby(['legal_id'])['canal', 'fecha'].max().reset_index()
df_last_click.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3674 entries, 0 to 3673
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype              
---  ------    --------------  -----              
 0   legal_id  3674 non-null   float64            
 1   canal     3674 non-null   object             
 2   fecha     3674 non-null   datetime64[ns, UTC]
dtypes: datetime64[ns, UTC](1), float64(1), object(1)
memory usage: 86.2+ KB


/var/folders/gt/z1ffyd2d3qdgxgy_874364xc0000gr/T/ipykernel_39095/1177465108.py:1: FutureWarning: Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.
  df_last_click = df_wompi.groupby(['legal_id'])['canal', 'fecha'].max().reset_index()


In [176]:
# Cambiar nombres
df_last_click.columns = ['num_doc', 'canal', 'fecha_lead']
df_last_click

,num_doc,canal,fecha_lead
0,1.730770e+05,google,2025-08-08 20:35:49.152000+00:00
1,2.790260e+05,google,2025-08-24 22:58:59.883000+00:00
2,3.325930e+05,google,2025-09-13 22:38:52.427000+00:00
3,3.332610e+05,google,2025-08-29 21:47:39.977000+00:00
4,3.904850e+05,google,2025-08-05 14:50:07.629000+00:00
...,...,...,...
3669,2.000018e+09,google,2025-09-09 21:26:55.993000+00:00
3670,2.000019e+09,google,2025-09-17 14:12:56.471000+00:00
3671,2.000019e+09,google,2025-10-14 19:42:22.261000+00:00
3672,3.202981e+09,google,2025-10-10 15:20:31.483000+00:00


In [177]:
# Seleccionar año, mes y dia de la fecha lead
df_last_click['fecha_lead'] = df_last_click.fecha_lead.apply(lambda x: pd.to_datetime(x).strftime('%Y-%m-%d'))

In [178]:
df_last_click.groupby('canal')['num_doc'].count().reset_index().sort_values(by='num_doc', ascending=False).reset_index(drop=True)

,canal,num_doc
0,google,3618
1,email,49
2,meta,7


In [187]:
# Subir a LZ
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_estrategia_5x_huella_digital_wompi4 PURGE;"""
helper.ejecutar_consulta(sql_drop)

sparky.subir_df(df_last_click, nombre_tabla='proceso.mdo_estrategia_5x_huella_digital_wompi4', zona='proceso_vdm')

sql_compute = "COMPUTE INCREMENTAL STATS proceso.mdo_estrategia_5x_huella_digital_wompi4;"
helper.ejecutar_consulta(sql_compute)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 31/31      DROP ...o_estrategia_5x_huella_digital_wompi4   finalizado   08:36:17 AM     00:00.2 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 32/32   DF A LZ ...o_estrategia_5x_huella_digital_wompi4   ejecutando   08:36:18 AM             

2026-02-10 08:36:23 - [INFO] - Intento 1 de 3


 32/32   DF A LZ ...o_estrategia_5x_huella_digital_wompi4   finalizado   08:36:18 AM     01:18.1 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 33/33   COMPUTE ...o_estrategia_5x_huella_digital_wompi4   finalizado   08:37:36 AM     00:01.5 
-------------------------------------------------------------------------------------------------


## Determinar Atribución

In [188]:
dict_ult_ing_wompi_merch = helper.obtener_ultima_ingestion('resultados_wompi.wompi_merchants')
dict_ult_ing_wompi_merch

2026-02-10 08:37:37 - [INFO] - Buscando fechas para resultados_wompi.wompi_merchants
/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:476: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, cn)
2026-02-10 08:37:38 - [INFO] - Finalizo la busqueda, duracion: 00:00.8, resultado: {'year': 2026, 'month': 2, 'day': 10}


{'year': 2026, 'month': 2, 'day': 10}

In [189]:
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_vinculacion_wompi PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_vinculacion_wompi STORED AS PARQUET AS
SELECT id_comercio,
       documento_identidad,
       tipo_documento,
       creado,
       to_timestamp(cast(creado AS string), 'yyyyMMdd') AS fecha_vinc
FROM resultados_wompi.wompi_merchants
WHERE YEAR = """ + str(dict_ult_ing_wompi_merch['year']) + """
  AND MONTH = """ + str(dict_ult_ing_wompi_merch['month']) + """
  AND DAY = """ + str(dict_ult_ing_wompi_merch['day']) + """
  AND modelo = 'Agregador'
  and activo = 'A'
  and desembolsos_permitidos = 'Si';
"""
helper.ejecutar_consulta(sql)

sql_compute = "COMPUTE INCREMENTAL STATS proceso.mdo_vinculacion_wompi;"
helper.ejecutar_consulta(sql_compute)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 34/34      DROP            proceso.mdo_vinculacion_wompi   finalizado   08:37:38 AM     00:00.2 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 35/35    CREATE            proceso.mdo_vinculacion_wompi   finalizado   08:37:39 AM     00:00.7 
-------------------------------------------------------------------------------------------------
--------------------

In [191]:
# Vinculaciones con leads
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_vinculacion_wompi_con_leads PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_vinculacion_wompi_con_leads STORED AS PARQUET AS WITH outcome1 AS
  (SELECT a.id_comercio,
          a.documento_identidad,
          a.tipo_documento,
          a.creado,
          a.fecha_vinc,
          b.fecha_lead,
          b.canal
   FROM proceso.mdo_vinculacion_wompi AS a
   LEFT JOIN proceso.mdo_estrategia_5x_huella_digital_wompi4 AS b ON cast(a.documento_identidad AS BIGINT) = cast(b.num_doc AS BIGINT)),
                                                                               outcome2 AS
  (SELECT a.id_comercio,
          a.documento_identidad,
          a.tipo_documento,
          a.creado,
          a.fecha_vinc,
          a.fecha_lead,
          a.canal,
          datediff(a.fecha_vinc, a.fecha_lead) AS tiempo_transcurrido
   FROM outcome1 AS a)
SELECT a.id_comercio,
       a.documento_identidad,
       a.tipo_documento,
       a.creado,
       a.fecha_vinc,
       a.fecha_lead,
       a.canal,
       a.tiempo_transcurrido,
       CASE
           WHEN a.tiempo_transcurrido >= 0 THEN 1
           ELSE 0
       END AS lead_valido
FROM outcome2 AS a;
"""
helper.ejecutar_consulta(sql)

sql_compute = "COMPUTE INCREMENTAL STATS proceso.mdo_vinculacion_wompi_con_leads;"
helper.ejecutar_consulta(sql_compute)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 39/39      DROP  proceso.mdo_vinculacion_wompi_con_leads   finalizado   08:39:55 AM     00:00.2 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 40/40    CREATE  proceso.mdo_vinculacion_wompi_con_leads   finalizado   08:39:56 AM     00:01.2 
-------------------------------------------------------------------------------------------------
--------------------

In [201]:
# Obtener atribución
sql_df = """
select count(*) as total_vinc
from proceso.mdo_vinculacion_wompi_con_leads
WHERE fecha_vinc BETWEEN '2025-08-01' AND '2026-02-02'
"""
df_total_vinc = helper.obtener_dataframe(sql_df)
df_total_vinc

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 47/47 DATAFRAME                                           descargando   03:06:18 PM             

2026-02-10 15:06:21 - [INFO] - 1 filas, 1 columnas, 00:02.2 consultando, 00:00.3 descargando, 00:00.0 convirtiendo


 47/47 DATAFRAME                                            finalizado   03:06:18 PM     00:02.8 
-------------------------------------------------------------------------------------------------


,total_vinc
0,66670


In [204]:
df_total_vinc.total_vinc.values[0]

66670

In [209]:
# Data frame para obtener atribución mercadeo y canal
sql_df = """
WITH outcome1 AS
  (SELECT nvl(canal, 'sin_canal') AS canal,
          count(*) AS num_vinc,
          """ + str(df_total_vinc.total_vinc.values[0]) + """ AS total_vinc
   FROM proceso.mdo_vinculacion_wompi_con_leads
   WHERE fecha_vinc BETWEEN '2025-08-01' AND '2026-02-02'
     AND lead_valido = 1
   GROUP BY 1)
SELECT canal,
       num_vinc,
       total_vinc,
       round(num_vinc/total_vinc, 5) AS prop
FROM outcome1
ORDER BY num_vinc DESC;
"""
df_atribucion = helper.obtener_dataframe(sql_df)
df_atribucion

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 49/49 DATAFRAME                                           descargando   03:20:04 PM             

2026-02-10 15:20:06 - [INFO] - 3 filas, 4 columnas, 00:00.8 consultando, 00:00.3 descargando, 00:00.0 convirtiendo


 49/49 DATAFRAME                                            finalizado   03:20:04 PM     00:01.4 
-------------------------------------------------------------------------------------------------


,canal,num_vinc,total_vinc,prop
0,google,2680,66670,0.04020
1,email,36,66670,0.00054
2,meta,3,66670,0.00004


In [208]:
# Atribución bruta mercadeo 
df_atribucion.num_vinc.sum()

2719

In [207]:
# Atribución mercadeo
round(df_atribucion.num_vinc.sum() / df_total_vinc.total_vinc.values[0], 4)

0.0408

# Borrar tablas proceso.

In [210]:
# Borrar tablas proceso.
tablas_borrar = [
    'proceso.mdo_estrategia_5x_huella_digital_wompi4',
    'proceso.mdo_vinculacion_wompi',
    'proceso.mdo_vinculacion_wompi_con_leads'
]
for tabla in tablas_borrar:
    sql_drop = f"""DROP TABLE IF EXISTS {tabla} PURGE;"""
    helper.ejecutar_consulta(sql_drop)

2026-02-10 16:10:20 - [INFO] - Transcurrido: 3016, Tiempo de Refresco = 1000


-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 50/50      DROP ...o_estrategia_5x_huella_digital_wompi4   finalizado   04:10:21 PM     00:00.5 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 51/51      DROP            proceso.mdo_vinculacion_wompi   finalizado   04:10:22 PM     00:00.4 
-------------------------------------------------------------------------------------------------
--------------------